# PneumoScan — Explainable Pneumonia Screening on Adult Chest X-rays

**Dataset:** RSNA Pneumonia Detection Challenge (adult, radiologist-annotated bounding boxes)
**Task:** binary — NORMAL vs PNEUMONIA (lung opacity)
**Pipeline:** DICOM → CLAHE → augmentation → DenseNet121 / EfficientNet-B0 → Grad-CAM **scored against the boxes** → TFLite

> **Set the runtime first:** `Runtime → Change runtime type → T4 GPU`.
> **And accept the competition rules once:** https://www.kaggle.com/competitions/rsna-pneumonia-detection-challenge/rules
> — click *I Understand and Accept*, or every download returns 403.

A full run is **90–120 minutes**. Section 5 offers a 20-minute rehearsal on a
subsample first — do that before committing to the long run.

---
1. GPU check  2. Project code  3. Kaggle credentials  4. Download + CLAHE + splits
5. Rehearsal (optional)  6. Train DenseNet121  7. Train EfficientNet-B0
8. Compare  9. Grad-CAM + localisation  10. TFLite  11. Download results

## 1. Check the GPU

In [ ]:
!nvidia-smi -L
!pip install -q pydicom
import tensorflow as tf, keras, pydicom
print("TensorFlow", tf.__version__, "| Keras", keras.__version__, "| pydicom", pydicom.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPU:", gpus or "NONE  -> Runtime > Change runtime type > T4 GPU, then rerun")

## 2. Get the project code

Pick **one**: 2a if the project is on GitHub, 2b to upload `pneumoscan.zip`.

In [ ]:
# --- 2a. Clone from GitHub -------------------------------------------------
REPO_URL = ""   # e.g. "https://github.com/<you>/pneumoscan.git"

import os, shutil
if REPO_URL:
    shutil.rmtree("/content/project", ignore_errors=True)
    !git clone -q $REPO_URL /content/project
    os.chdir("/content/project")
    print("cloned into", os.getcwd())
    !ls
else:
    print("REPO_URL is empty - use cell 2b instead.")

In [ ]:
# --- 2b. Upload pneumoscan.zip ---------------------------------------------
import os, glob, zipfile, shutil
from google.colab import files

up = files.upload()
name = next(iter(up))
shutil.rmtree("/content/project", ignore_errors=True)
shutil.rmtree("/content/_unzip", ignore_errors=True)
os.makedirs("/content/project", exist_ok=True)
with zipfile.ZipFile(name) as z:
    z.extractall("/content/_unzip")

root = next(iter(glob.glob("/content/_unzip/**/src/train.py", recursive=True)), None)
assert root, "src/train.py not found inside the zip"
src_root = os.path.dirname(os.path.dirname(root))
for item in os.listdir(src_root):
    shutil.move(os.path.join(src_root, item), "/content/project")
os.chdir("/content/project")
print("project at", os.getcwd())
!ls

## 3. Kaggle credentials

Kaggle → avatar → **Settings** → **API** → *Create New Token* downloads `kaggle.json`.

Preferred: add `KAGGLE_USERNAME` and `KAGGLE_KEY` to Colab **Secrets** (🔑 in the
sidebar) with notebook access enabled. Otherwise the cell falls back to uploading
the file.

In [ ]:
import os, json, pathlib

try:
    from google.colab import userdata
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
    print("using Colab secrets for", os.environ["KAGGLE_USERNAME"])
except Exception as e:
    print("secrets unavailable (%s) - upload kaggle.json instead" % type(e).__name__)
    from google.colab import files
    up = files.upload()
    creds = json.loads(next(iter(up.values())))
    os.environ["KAGGLE_USERNAME"], os.environ["KAGGLE_KEY"] = creds["username"], creds["key"]

d = pathlib.Path.home() / ".kaggle"; d.mkdir(exist_ok=True)
(d / "kaggle.json").write_text(json.dumps({
    "username": os.environ["KAGGLE_USERNAME"], "key": os.environ["KAGGLE_KEY"]}))
(d / "kaggle.json").chmod(0o600)
print("kaggle.json ready")

## 4. Download, CLAHE, and split  *(~15 min)*

`scripts/prepare_data.py --download`:

* pulls 3.7 GB of DICOMs from Kaggle and unzips them,
* collapses the per-box CSVs into one row per image, carrying the boxes,
* splits 70/15/15, stratified on the label and grouped by patient,
* applies **CLAHE** and writes lossless 224×224 PNGs,
* rescales every radiologist box into the resized frame,
* reads DICOM headers for a cohort table — which is how you *show* the data is adult.

**If this fails with 403**, you have not accepted the competition rules yet:
https://www.kaggle.com/competitions/rsna-pneumonia-detection-challenge/rules

In [ ]:
!pip install -q kaggle
!python scripts/prepare_data.py --download

In [ ]:
import pandas as pd
from IPython.display import Image, display

print("Split sizes");     display(pd.read_csv("reports/dataset_summary.csv"))
print("RSNA classes");    display(pd.read_csv("reports/detailed_class_summary.csv", index_col=0))
print("Cohort (adult?)"); display(pd.read_csv("reports/cohort.csv"))
display(Image("reports/clahe_examples.png", width=760))

### Sanity check

Roughly what you should see across all splits combined:

| | images |
|---|---|
| Lung Opacity (label 1) | ~6,000 |
| Normal | ~8,850 |
| No Lung Opacity / Not Normal | ~11,800 |
| **total** | **26,684** |

and about 9,500 boxes. In the cohort table, `pct_under_18` should be near zero
and the median age around 50 — that is your evidence the dataset is adult.

Note the third class: films that are abnormal for some reason *other* than
pneumonia. They are kept as negatives, because a screening tool has to tell
pneumonia from other pathology, not just from healthy lungs. Pass
`--exclude-not-normal` if you want to measure how much easier the task gets
without them.

## 5. Rehearsal on a subsample *(optional, ~20 min — recommended)*

Runs the entire pipeline on 3,000 training images so you find any problem in 20
minutes instead of 2 hours. Validation and test stay full size, so the numbers
are honest, just weaker. Skip to section 6 if you would rather go straight in.

In [ ]:
!python src/train.py --backbone densenet121 --tag rehearsal --subsample-train 3000 --head-epochs 1 --finetune-epochs 2
!python scripts/make_gradcam_figures.py --backbone densenet121 --tag rehearsal --loc-max 300

## 6. Train DenseNet121  *(~35 min)*

In [ ]:
!python src/train.py --backbone densenet121

## 7. Train EfficientNet-B0  *(~30 min, optional)*

**You do not need this to have a complete project.** One trained backbone runs
the whole pipeline through to TFLite. Skip straight to section 8 if you are short
on time or worried about the session dropping — everything downstream adapts to
however many models exist.

What the second one buys you: a real comparison table, and the deployment
argument (EfficientNet-B0 is ~4M parameters against DenseNet121's ~7M, and
quantises to roughly 4.5 MB against 7.4 MB — directly relevant to the mobile
objective). It also reproduces the Khadidos et al. comparison you cite, on adult
data instead of paediatric.

In [ ]:
!python src/train.py --backbone efficientnetb0

## 8. Compare the two backbones

Pick the winner on **validation** AUROC. The test set is reported, never selected on.

In [ ]:
import json, pandas as pd
from pathlib import Path

rows = []
for f in sorted(Path("reports").glob("*/metrics.json")):
    if "rehearsal" in f.parent.name:
        continue
    m = json.loads(f.read_text())
    lo, hi = m["test_auroc_95ci"]
    rows.append({"run": f.parent.name,
                 "val AUROC": round(m["val"]["auroc"], 4),
                 "test AUROC": round(m["test"]["auroc"], 4),
                 "95% CI": "%.3f-%.3f" % (lo, hi),
                 "accuracy": round(m["test"]["accuracy"], 4),
                 "sensitivity": round(m["test"]["sensitivity_recall"], 4),
                 "specificity": round(m["test"]["specificity"], 4),
                 "F1": round(m["test"]["f1"], 4),
                 "min": m.get("train_minutes")})
comparison = pd.DataFrame(rows).sort_values("val AUROC", ascending=False)
comparison.to_csv("reports/backbone_comparison.csv", index=False)
comparison

In [ ]:
from IPython.display import Image, display
BEST = comparison.iloc[0]["run"]
print("best run:", BEST)
for f in ("training_curves.png", "test_curves.png", "test_confusion.png"):
    display(Image(f"reports/{BEST}/{f}", width=820))

## 9. Grad-CAM — and the number that makes it evidence

RSNA gives radiologist-drawn boxes, so the heatmaps can be *scored* rather than
admired:

* **pointing game** — does the hottest pixel land inside a box?
* **energy pointing game** — what share of the heatmap's mass is inside the boxes?
* **IoU@0.5** — overlap after thresholding at half the peak.

Each is reported against a **chance baseline** (the boxes' own share of the
image). A pointing-game score means nothing without it — that ratio, the *lift*,
is the number to quote.

In [ ]:
!python scripts/make_gradcam_figures.py --backbone {BEST}

In [ ]:
import pandas as pd
from IPython.display import Image, display

loc = pd.read_csv(f"reports/{BEST}/localization.csv")
display(loc[["explainer", "n", "pointing_game", "pointing_game_chance",
             "pointing_game_lift", "energy_pointing_game"]])

for f in ("gradcam_pneumonia.png", "gradcam_errors.png",
          "gradcam_vs_pp.png", "gradcam_normal.png"):
    try:
        display(Image(f"reports/{BEST}/{f}", width=900))
    except Exception:
        print("missing:", f)

## 10. Export to TensorFlow Lite

Writes a float32 and a dynamic-range int8 model, and refuses to ship the float
one if it drifts from Keras by more than 1e-3 on real test images.

In [ ]:
from pathlib import Path

# Export whichever backbones actually got trained - skipping section 7 is fine.
trained = sorted(c.stem for c in Path("checkpoints").glob("*.keras")
                 if c.stem in ("densenet121", "efficientnetb0"))
print("exporting:", trained or "nothing found - train a model first")
for b in trained:
    !python src/export_tflite.py --backbone {b}

## 11. Download the results

In [ ]:
!zip -qr /content/results.zip reports checkpoints -x "*.npz"
from google.colab import files
files.download("/content/results.zip")

In [ ]:
# Safer on a flaky connection: keep everything on Drive as you go.
# from google.colab import drive; drive.mount("/content/drive")
# !mkdir -p "/content/drive/MyDrive/pneumoscan" && cp -r reports checkpoints "/content/drive/MyDrive/pneumoscan/"